# Gemini API로 식단 재료·알레르기 추정

`Crawling.ipynb`와 동일한 방식으로 금오공대 식당 메뉴를 가져온 뒤, **Google Gemini**에 메뉴 문구를 보내 **추정 재료**와 **알레르기 유발 가능 식품**(식약처 고시 기준 등)을 정리합니다.

> **주의**: 모델 출력은 **추정**이며, 실제 조리·원산지·레시피와 다를 수 있습니다. 알레르기가 있으면 **반드시 식당에 직접 확인**하세요.

## API 키

1. [Google AI Studio](https://aistudio.google.com/apikey)에서 API 키 발급  
2. 터미널에서 `export GEMINI_API_KEY='여기에_키'` 후 커널 재시작  
3. **429 / `free_tier_requests` 한도**: 무료 티어가 막히면 [AI Studio](https://aistudio.google.com/apikey)에서 **유료 결제가 연결된 프로젝트**의 API 키를 쓰거나, `GEMINI_MODEL`을 바꿔 보세요.  
4. API 키는 **노트북/깃에 절대 커밋하지 마세요.** 유출 시 AI Studio에서 키를 **삭제 후 재발급**하세요.

In [1]:
# 최초 1회 또는 ImportError 시 실행
%pip install -q google-genai pandas requests lxml
# (선택) 노트북 진행바 경고 줄이기: %pip install -q ipywidgets

You should consider upgrading via the '/Applications/Xcode.app/Contents/Developer/usr/bin/python3 -m pip install --upgrade pip' command.
Note: you may need to restart the kernel to use updated packages.


In [2]:
import json
import os
import re
from io import StringIO

import pandas as pd
import requests
from google import genai
from google.genai import types

# API 키: 환경변수 GEMINI_API_KEY 권장 (노트북에 키를 직접 쓰지 마세요)
api_key = os.environ["GEMINI_API_KEY"] = "REMOVED"
if not api_key:
    raise RuntimeError(
        "환경변수 GEMINI_API_KEY가 없습니다. "
        "터미널: export GEMINI_API_KEY='...' 후 커널 재시작, "
        "또는 이 셀에서 os.environ['GEMINI_API_KEY'] = '...' 설정"
    )

# 모델명은 환경에 따라 조정
MODEL_NAME = "gemini-2.5-flash-lite"
client = genai.Client(api_key=api_key)
print("사용 모델:", MODEL_NAME)

/Users/gimhyeonjin/Library/Python/3.9/lib/python/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(
/Users/gimhyeonjin/Library/Python/3.9/lib/python/site-packages/google/auth/__init__.py:54: FutureWarning: You are using a Python version 3.9 past its end of life. Google will update google-auth with critical bug fixes on a best-effort basis, but not with any other fixes or features. Please upgrade your Python version, and then update google-auth.
  warnings.warn(eol_message.format("3.9"), FutureWarning)
/Users/gimhyeonjin/Library/Python/3.9/lib/python/site-packages/google/oauth2/__init__.py:40: FutureWarning: You are using a Python version 3.9 past its end of life. Google will update google-auth with critical bug fixes on a best-effort basis, but not with any other fixes or features. Please upgrade your Python ve

사용 모델: gemini-2.5-flash-lite


In [3]:
# Crawling.ipynb와 동일: 식단표 수집
URLS = {
    "학생식당": "https://www.kumoh.ac.kr/ko/restaurant01.do",
    "교직원식당": "https://www.kumoh.ac.kr/ko/restaurant02.do",
    "분식당": "https://www.kumoh.ac.kr/ko/restaurant04.do",
}


def fetch_html(url: str) -> str:
    res = requests.get(url, timeout=15)
    res.raise_for_status()
    res.encoding = "utf-8"
    return res.text


def load_menus() -> dict[str, pd.DataFrame]:
    menus: dict[str, pd.DataFrame] = {}
    for name, url in URLS.items():
        html = fetch_html(url)
        tables = pd.read_html(StringIO(html))
        if not tables:
            continue
        df = tables[0].copy()
        df.columns = [str(c).strip() for c in df.columns]
        df = df.replace(r"\s+", " ", regex=True)
        menus[name] = df
    return menus


menus = load_menus()
for k, v in menus.items():
    print(k, v.shape)

학생식당 (2, 7)
교직원식당 (2, 7)
분식당 (1, 7)


In [4]:
SYSTEM_INSTRUCTION = """당신은 한국 대학 급식 메뉴 문구를 분석하는 조력자입니다.
메뉴 이름·반찬 나열에서 **추정되는 재료**와 **알레르기 유발 가능이 있는 식품**을 정리합니다.

규칙:
- 반드시 **한국어**로 답합니다.
- 메뉴 문구만 보고 **추정**이며, 확정이 아님을 전제로 합니다.
- 알레르기는 식약처 고시 **알레르기 유발 식품 표시 대상**(난류, 우유, 메밀, 땅콩, 대두, 밀, 고등어, 게, 새우, 돼지고기, 복숭아, 토마토, 아황산류, 호두, 닭고기, 쇠고기, 오징어, 조개류, 잣 등) 중 **해당될 가능성이 있는 것**을 골라 이유를 짧게 씁니다.
- 없으면 빈 배열로 둡니다.
- 응답은 **JSON만** 출력합니다. 마크다운 코드블록 금지."""

import time

from google.api_core import exceptions as google_api_exceptions


def iter_menu_entries(menus: dict) -> list[dict]:
    """DataFrame 셀 단위로 분석할 메뉴 문구 목록."""
    out = []
    for place, df in menus.items():
        for col in df.columns:
            for idx, val in df[col].items():
                s = str(val).strip()
                if not s or "운영 없음" in s:
                    continue
                out.append(
                    {
                        "식당": place,
                        "요일열": col,
                        "표행": int(idx),
                        "메뉴텍스트": s,
                    }
                )
    return out


def extract_json_array(text: str) -> list:
    """모델이 JSON 배열만 반환하도록 했을 때 파싱."""
    text = text.strip()
    m = re.search(r"\[[\s\S]*\]", text)
    if not m:
        raise ValueError(f"JSON 배열을 찾을 수 없습니다: {text[:200]}...")
    return json.loads(m.group())


def analyze_menus_with_gemini(
    entries: list[dict],
    batch_size: int = 4,
    sleep_between_batches_sec: float = 21.0,
    max_retries: int = 5,
) -> list[dict]:
    """여러 메뉴를 묶어 한 번에 분석. 429(할당량) 시 대기 후 재시도."""
    all_results: list[dict] = []
    for i in range(0, len(entries), batch_size):
        chunk = entries[i : i + batch_size]
        user_payload = json.dumps(chunk, ensure_ascii=False)
        prompt = f"""다음은 급식표에서 뽑은 객체 배열입니다. 각 항목에 대해 분석해 주세요.

입력:
{user_payload}

출력 형식: **JSON 배열** 하나만. 각 원소는 입력 순서와 동일하게 다음 키를 가집니다.
- "식당", "요일열", "표행", "메뉴텍스트" (입력과 동일하게 복사)
- "추정_재료": 문자열 배열 (예: ["쌀", "닭고기", ...])
- "알레르기_유발가능": 배열. 각 원소는 {{"식품": "...", "근거": "메뉴 문구에서 ... 때문에"}} 형태

다른 설명 없이 JSON 배열만 출력하세요."""

        resp = None
        last_err = None
        for attempt in range(max_retries):
            try:
                resp = client.models.generate_content(
                    model=MODEL_NAME,
                    contents=[SYSTEM_INSTRUCTION, prompt],
                    config=types.GenerateContentConfig(
                        temperature=0.2,
                        max_output_tokens=8192,
                    ),
                )
                break
            except google_api_exceptions.ResourceExhausted as e:
                last_err = e
                wait = sleep_between_batches_sec * (attempt + 1)
                print(
                    f"429 할당량/속도 제한 — {wait:.0f}초 대기 후 재시도 ({attempt + 1}/{max_retries})"
                )
                time.sleep(wait)
        if resp is None:
            raise last_err  # type: ignore[misc]

        raw = (getattr(resp, "text", "") or "").strip()
        if not raw:
            raise RuntimeError("Gemini 응답이 비었습니다. finish_reason 또는 안전 필터를 확인하세요.")
        parsed = extract_json_array(raw)
        if len(parsed) != len(chunk):
            print(
                f"경고: 배치 {i // batch_size}에서 개수 불일치 (기대 {len(chunk)}, 실제 {len(parsed)})"
            )
        for j, e in enumerate(chunk):
            if j < len(parsed):
                row = parsed[j]
                row.setdefault("메뉴텍스트", e["메뉴텍스트"])
                all_results.append(row)
            else:
                all_results.append(
                    {
                        **e,
                        "추정_재료": [],
                        "알레르기_유발가능": [],
                        "오류": "모델 응답에서 해당 항목 누락",
                    }
                )
        if i + batch_size < len(entries):
            time.sleep(sleep_between_batches_sec)
    return all_results


entries = iter_menu_entries(menus)
print("분석 대상 셀 수:", len(entries))
entries[:2]

분석 대상 셀 수: 25


/Users/gimhyeonjin/Library/Python/3.9/lib/python/site-packages/google/api_core/_python_version_support.py:246: FutureWarning: You are using a non-supported Python version (3.9.6). Google will not post any further updates to google.api_core supporting this Python version. Please upgrade to the latest Python version, or at least Python 3.10, and then update google.api_core.
  warnings.warn(message, FutureWarning)


[{'식당': '학생식당',
  '요일열': '월(03.30)',
  '표행': 0,
  '메뉴텍스트': '조식 [천원의 아침밥] *재학생만 해당 08:20~10:00 든든한제육&돈까스도시락'},
 {'식당': '학생식당',
  '요일열': '월(03.30)',
  '표행': 1,
  '메뉴텍스트': '중식 [정식: 4500원] 11:30~13:30 김밥맛볶음밥 팽이버섯된장국 김구이 계란장조림 실곤약야채무침 맛김치'}]

In [5]:
# 실제 API 호출 (항목이 많으면 시간·토큰 소모)
# 429 / free_tier quota: 유료 결제 연동 API 키 사용 또는 entries[:4]로 먼저 테스트
results = analyze_menus_with_gemini(entries)
# results = analyze_menus_with_gemini(entries[:4])

flat_rows = []
for r in results:
    ingredients = r.get("추정_재료", [])
    allergies = r.get("알레르기_유발가능", [])
    flat_rows.append(
        {
            "식당": r.get("식당"),
            "요일열": r.get("요일열"),
            "표행": r.get("표행"),
            "메뉴텍스트": r.get("메뉴텍스트"),
            "추정_재료": ", ".join(ingredients) if isinstance(ingredients, list) else str(ingredients),
            "알레르기_요약": " / ".join(
                f"{a.get('식품', a)} — {a.get('근거', '')}" if isinstance(a, dict) else str(a)
                for a in (allergies or [])
            ),
        }
    )

summary_df = pd.DataFrame(flat_rows)
pd.set_option("display.max_colwidth", None)
display(summary_df)

,식당,요일열,표행,메뉴텍스트,추정_재료,알레르기_요약
0,학생식당,월(03.30),0,조식 [천원의 아침밥] *재학생만 해당 08:20~10:00 든든한제육&돈까스도시락,"돼지고기, 닭고기, 밀가루, 쌀",돼지고기 — 제육 / 닭고기 — 돈까스 (튀김옷에 닭고기 사용 가능성) / 밀 — 돈까스 (튀김옷)
1,학생식당,월(03.30),1,중식 [정식: 4500원] 11:30~13:30 김밥맛볶음밥 팽이버섯된장국 김구이 계란장조림 실곤약야채무침 맛김치,"쌀, 계란, 대두, 밀, 팽이버섯, 김, 곤약, 채소","계란 — 계란장조림 / 대두 — 된장국 (된장), 간장 (장조림, 볶음밥) / 밀 — 김밥맛볶음밥 (소스류에 포함 가능성), 간장 (장조림, 볶음밥)"
2,학생식당,화(03.31),0,조식 [천원의 아침밥] *재학생만 해당 08:20~10:00 새콤달콤유부초밥 에그치즈토마토샌드위치 홈요거트,"쌀, 유부, 계란, 치즈, 토마토, 밀, 우유","계란 — 에그치즈토마토샌드위치 / 토마토 — 에그치즈토마토샌드위치 / 밀 — 샌드위치 (빵) / 우유 — 치즈, 홈요거트"
3,학생식당,화(03.31),1,중식 [정식: 4500원] 11:30~13:30 제육덮밥 무채맑은국 미니메밀전병구이 콩나물무침 맛김치,"돼지고기, 쌀, 메밀, 콩나물, 채소",돼지고기 — 제육덮밥 / 메밀 — 미니메밀전병구이
4,학생식당,수(04.01),0,조식 [천원의 아침밥] *재학생만 해당 08:20~10:00 콰트로고기정찬도시락,"돼지고기, 소고기, 닭고기, 쌀",돼지고기 — 한돈(돼지고기)이 포함될 가능성 / 소고기 — 고기 정찬에 소고기가 포함될 가능성 / 닭고기 — 고기 정찬에 닭고기가 포함될 가능성
5,학생식당,수(04.01),1,중식 [정식: 4500원] 11:30~13:30 쌀밥 닭곰탕 연두부/양념장 볶음멸치조림 마늘종장아찌무침 깍두기,"쌀, 닭고기, 두부, 멸치, 마늘, 무",닭고기 — 닭곰탕에 닭고기가 포함됨 / 대두 — 연두부에 대두가 포함됨 / 멸치 — 볶음멸치조림에 멸치가 포함됨 / 쇠고기 — 양념장에 쇠고기가 포함될 가능성
6,학생식당,목(04.02),0,조식 [천원의 아침밥] *재학생만 해당 08:20~10:00 한돈가츠동덮밥도시락 딸기요플레,"돼지고기, 쌀, 딸기, 우유",돼지고기 — 한돈가츠에 돼지고기가 포함됨 / 밀 — 가츠(돈까스) 튀김옷에 밀가루가 사용될 가능성 / 대두 — 덮밥 소스에 대두(간장 등)가 포함될 가능성 / 딸기 — 딸기 요플레에 딸기가 포함됨 / 우유 — 요플레에 우유가 포함됨
7,학생식당,목(04.02),1,중식 [정식: 4500원] 11:30~13:30 쌀밥 돼지찌개 감자크로켓 오이생채 어묵채볶음 맛김치,"쌀, 돼지고기, 감자, 오이, 어묵, 배추",돼지고기 — 돼지찌개에 돼지고기가 포함됨 / 밀 — 감자크로켓 튀김옷에 밀가루가 사용될 가능성 / 대두 — 어묵에 대두가 포함될 가능성 / 쇠고기 — 맛김치 양념에 쇠고기가 포함될 가능성
8,학생식당,금(04.03),0,조식 [천원의 아침밥] *재학생만 해당 08:20~10:00 다찬스페셜정식도시락 요구르트,"쌀, 다양한 반찬, 요구르트",우유 — 요구르트 때문에 / 대두 — 다찬스페셜정식도시락에 포함될 가능성 때문에 / 밀 — 다찬스페셜정식도시락에 포함될 가능성 때문에
9,학생식당,금(04.03),1,중식 [정식: 4500원] 11:30~13:30 쌀밥 짬뽕순두부찌개 한입돈까스 다시마부각 쥐어채볶음 깍두기,"쌀, 순두부, 돼지고기, 다시마, 쥐치, 채소, 배추",돼지고기 — 한입돈까스 때문에 / 밀 — 한입돈까스 튀김옷 때문에 / 대두 — 한입돈까스 소스 또는 순두부찌개 때문에 / 고등어 — 쥐어채볶음에 포함될 가능성 때문에 / 조개류 — 짬뽕순두부찌개에 포함될 가능성 때문에


In [6]:
# 선택: 결과를 CSV로 저장
summary_df.to_csv("menu_allergy_gemini.csv", index=False, encoding="utf-8-sig")